# Signature detection: build a 50k synthetic dataset and train YOLOv8 in Colab

**Before running:** make a Google Drive folder `MyDrive/signature_project` and upload:
`signatures.zip`, `best.pt`, `test_signature_document.pdf`, `get_pages.py`, `make_dataset.py`, `detect.py`

**Run it in two phases (saves your free GPU time):**
1. **Build data on CPU:** Runtime → Change runtime type → **CPU** → Run all. Steps 1–5 download pages and generate 50k images (~3–4 h), then save the dataset zip to Drive. Step 6 stops with a "switch to GPU" message.
2. **Train on GPU:** Runtime → Change runtime type → **T4 GPU** → Run all. The dataset is restored from Drive in a few minutes and training starts.

**Dataset built on your PC?** Upload it as `signature_dataset_50k.zip` to the same folder. Steps 3–4 then only unzip it, and the GPU is used just for training.

**If Colab disconnects:** run all again. The finished dataset is restored from Drive and training resumes from the last epoch.
Only a disconnect *during* page download / generation (steps 3–4, about 2–3 h) has to restart those steps.

In [ ]:
# 1. Mount Drive and check the uploaded files
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess
DRIVE = "/content/drive/MyDrive/signature_project"
N_IMAGES = 50000
DATA_ZIP = f"{DRIVE}/signature_dataset_{N_IMAGES // 1000}k.zip"
RUN_DIR = f"{DRIVE}/runs"
RUN_NAME = "sig50k"

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

need = ["signatures.zip", "best.pt", "test_signature_document.pdf", "get_pages.py", "make_dataset.py", "detect.py"]
missing = [f for f in need if not os.path.exists(f"{DRIVE}/{f}")]
print("Missing:", missing or "nothing")
assert not missing, "Upload the missing files to " + DRIVE
for f in ["get_pages.py", "make_dataset.py", "detect.py"]:
    shutil.copy(f"{DRIVE}/{f}", "/content/")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime: fine for steps 1-5)")

In [ ]:
# 2. Install packages
sh("pip -q install ultralytics datasets pymupdf")

In [ ]:
# 3. Dataset: restore from Drive if it was already built, otherwise download pages
os.chdir("/content")
HAVE_DATA = os.path.exists(DATA_ZIP)
if HAVE_DATA:
    print("Restoring dataset from Drive (a few minutes)...")
    sh(f'unzip -q -o "{DATA_ZIP}" -d /content')
    # the zip may have been built on another PC, so point data.yaml at the Colab path
    with open("/content/dataset/data.yaml", "w") as f:
        f.write("path: /content/dataset\ntrain: images/train\nval: images/val\nnames:\n  0: signature\n")
else:
    # ~1-2 h. Re-running this cell in the same session continues where it stopped.
    sh(f"python get_pages.py --count {N_IMAGES} --out /content/pages")

In [ ]:
# 4. Generate the 50k images (skipped if restored) and back the dataset up to Drive
FONTS = [
    "ofl/greatvibes/GreatVibes-Regular.ttf", "ofl/allura/Allura-Regular.ttf",
    "ofl/sacramento/Sacramento-Regular.ttf", "apache/homemadeapple/HomemadeApple-Regular.ttf",
    "ofl/alexbrush/AlexBrush-Regular.ttf", "ofl/parisienne/Parisienne-Regular.ttf",
    "ofl/pinyonscript/PinyonScript-Regular.ttf", "ofl/mrdafoe/MrDafoe-Regular.ttf",
    "ofl/herrvonmuellerhoff/HerrVonMuellerhoff-Regular.ttf", "ofl/italianno/Italianno-Regular.ttf",
    "apache/yellowtail/Yellowtail-Regular.ttf", "ofl/kristi/Kristi-Regular.ttf",
    "ofl/labelleaurore/LaBelleAurore.ttf", "ofl/reeniebeanie/ReenieBeanie.ttf",
    "ofl/monsieurladoulaise/MonsieurLaDoulaise-Regular.ttf", "ofl/marckscript/MarckScript-Regular.ttf",
    "ofl/meddon/Meddon.ttf", "ofl/qwigley/Qwigley-Regular.ttf",
]
if not HAVE_DATA:
    os.makedirs("/content/fonts", exist_ok=True)
    for f in FONTS:
        url = "https://github.com/google/fonts/raw/main/" + f
        r = subprocess.run(f'wget -q -O "/content/fonts/{os.path.basename(f)}" "{url}"', shell=True)
        if r.returncode != 0:
            print("font not found (skipped):", f)
            os.remove(f"/content/fonts/{os.path.basename(f)}")
    print(len(os.listdir("/content/fonts")), "fonts ready")

    sh("unzip -q -o /content/drive/MyDrive/signature_project/signatures.zip -d /content")
    sh("python make_dataset.py --pages /content/pages --signatures /content/signatures "
       "--fonts /content/fonts --out /content/dataset")
    print("Backing up dataset to Drive (~10 min)...")
    sh(f'cd /content && zip -q -0 -r "{DATA_ZIP}.tmp" dataset && mv "{DATA_ZIP}.tmp" "{DATA_ZIP}"')

In [ ]:
# 5. Look at a few generated images with their label boxes (green = labelled signature)
import cv2, glob, random
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 7))
for ax, p in zip(axes, random.sample(glob.glob("/content/dataset/images/train/*.jpg"), 4)):
    img = cv2.imread(p); H, W = img.shape[:2]
    for line in open(p.replace("images", "labels").replace(".jpg", ".txt")).read().split("\n"):
        if line.strip():
            _, xc, yc, w, h = map(float, line.split())
            cv2.rectangle(img, (int((xc - w / 2) * W), int((yc - h / 2) * H)),
                          (int((xc + w / 2) * W), int((yc + h / 2) * H)), (0, 200, 0), 3)
    ax.imshow(img[:, :, ::-1]); ax.axis("off")
plt.show()
print(len(glob.glob("/content/dataset/images/train/*.jpg")), "train |",
      len(glob.glob("/content/dataset/images/val/*.jpg")), "val")

In [ ]:
# 6. Train (fine-tunes your best.pt). Checkpoints are written to Drive every epoch.
# ~10-13 min/epoch on a T4 -> ~7-9 h for 40 epochs. If disconnected: Run all again, it resumes.
from ultralytics import YOLO
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Dataset is saved in Drive. Now switch to Runtime > Change runtime type > T4 GPU and Run all.")

last = f"{RUN_DIR}/{RUN_NAME}/weights/last.pt"
if os.path.exists(last):
    print("Resuming from", last)
    YOLO(last).train(resume=True)
else:
    YOLO(f"{DRIVE}/best.pt").train(
        data="/content/dataset/data.yaml",
        epochs=40, imgsz=640, batch=32, workers=2,
        patience=10, close_mosaic=5, fliplr=0.0,
        project=RUN_DIR, name=RUN_NAME, exist_ok=True,
    )

In [ ]:
# 7. Validate and test on the real PDF
best = f"{RUN_DIR}/{RUN_NAME}/weights/best.pt"
YOLO(best).val(data="/content/dataset/data.yaml", imgsz=640)

sh(f'python detect.py "{DRIVE}/test_signature_document.pdf" --model "{best}" --out /content/output')
pages = sorted(glob.glob("/content/output/page_*.jpg"), key=lambda p: int(p.split("_")[-1][:-4]))
fig, axes = plt.subplots(2, 5, figsize=(25, 13))
for ax, p in zip(axes.flat, pages):
    ax.imshow(cv2.imread(p)[:, :, ::-1]); ax.set_title(os.path.basename(p)); ax.axis("off")
plt.tight_layout(); plt.show()

shutil.copy(best, f"{DRIVE}/best_50k.pt")
print("Saved model to", f"{DRIVE}/best_50k.pt")